# Bivariate Choropleth: NTL Radiance × Population Density

**Author:** Bouchra Daddaoui  
**Repository:** viirs-electrification-ml  

A **bivariate choropleth** encodes two variables simultaneously using a 3×3 colour matrix,
revealing their spatial co-variation in a single map — a technique common in demographic
and public health cartography, applied here to electrification inequality research.

**Variables:**
- **X-axis (columns):** NTL radiance — electrification intensity
- **Y-axis (rows):** Population density — demographic exposure

**Key quadrant interpretation:**
- High NTL / High pop → dense, electrified urban core
- Low NTL / High pop → dense but unelectrified (energy poor) — SDG 7 priority
- High NTL / Low pop → light industry or sparse but lit areas
- Low NTL / Low pop → remote, low-exposure areas

**Outputs:**
- `figures/bivariate_map.png` — publication-quality panel
- `figures/bivariate_summary_table.csv` — tile counts per class

In [ ]:
import sys
sys.path.insert(0, '../scripts')

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from bivariate import (
    assign_bivariate_class, plot_bivariate_panel,
    bivariate_summary, BIVARIATE_PALETTE, CLASS_LABELS,
)

FIGURES = Path('../figures')
FIGURES.mkdir(exist_ok=True)
print('Libraries loaded.')

## 1. Generate Synthetic Tile Data

Morocco bbox extended to 20.8°N to include Western Sahara.

In [ ]:
def generate_tiles(country, n_tiles, bbox, ntl_mean, ntl_std, seed):
    rng = np.random.default_rng(seed)
    minx, miny, maxx, maxy = bbox
    cols = int(np.sqrt(n_tiles))
    rows = n_tiles // cols
    xs = np.linspace(minx, maxx, cols + 1)
    ys = np.linspace(miny, maxy, rows + 1)
    geoms, tile_ids = [], []
    for i in range(rows):
        for j in range(cols):
            geoms.append(box(xs[j], ys[i], xs[j+1], ys[i+1]))
            tile_ids.append(f'{country[:3].upper()}_{i:02d}_{j:02d}')
    n = len(geoms)
    centroids = np.array([[g.centroid.x, g.centroid.y] for g in geoms])
    dists = np.linalg.norm(centroids[:, None] - centroids[None, :], axis=-1)
    cov = ntl_std**2 * np.exp(-dists / (0.3 * (maxx - minx)))
    ntl = rng.multivariate_normal(np.full(n, ntl_mean), cov)
    ntl = np.clip(ntl, 0, None)
    pop = rng.lognormal(mean=np.log(100), sigma=1.2, size=n)
    gdf = gpd.GeoDataFrame({
        'tile_id': tile_ids, 'country': country,
        'ntl_mean': ntl, 'pop_density': pop,
    }, geometry=geoms, crs='EPSG:4326')
    return gdf


countries_cfg = [
    dict(country='Brazil',  n_tiles=100, bbox=(-48,-23,-43,-18), ntl_mean=12.5, ntl_std=8.0,  seed=1),
    dict(country='China',   n_tiles=100, bbox=(116,29,122,33),   ntl_mean=28.3, ntl_std=14.0, seed=2),
    dict(country='Morocco', n_tiles=100, bbox=(-17.1,20.8,-1.0,35.9), ntl_mean=8.1, ntl_std=5.5, seed=3),
]

gdfs = {cfg['country']: generate_tiles(**cfg) for cfg in countries_cfg}

for c, g in gdfs.items():
    print(f'{c}: NTL [{g.ntl_mean.min():.1f}, {g.ntl_mean.max():.1f}] | '
          f'Pop [{g.pop_density.min():.0f}, {g.pop_density.max():.0f}]')

## 2. Bivariate Classification (3×3 Quantile)

Each variable is split into 3 equal-count bins (terciles).  
The 9 bivariate classes use the Cindy Brewer palette — widely used in academic cartography.

In [ ]:
gdfs_biv = {c: assign_bivariate_class(g) for c, g in gdfs.items()}

for country, gdf in gdfs_biv.items():
    print(f'\n{country}:')
    print(gdf['biv_label'].value_counts().to_string())

## 3. Publication-Quality Bivariate Panel Map

In [ ]:
fig = plot_bivariate_panel(
    gdfs_biv,
    ntl_col='ntl_mean',
    pop_col='pop_density',
    save_path=FIGURES / 'bivariate_map.png',
)
plt.show()

## 4. Bivariate Summary Table

Tile counts and percentages per bivariate class — useful for cross-country comparison.

In [ ]:
summary_tables = [bivariate_summary(g, c) for c, g in gdfs_biv.items()]
summary_df = pd.concat(summary_tables, ignore_index=True)

print(summary_df.to_string(index=False))

summary_df.to_csv(FIGURES / 'bivariate_summary_table.csv', index=False)
print('\nSaved: figures/bivariate_summary_table.csv')

## 5. Energy-Poor Hotspot Analysis

The critical SDG 7 quadrant: **high population density + low NTL** — areas with large
unelectrified populations. These tiles represent priority intervention zones.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
hotspot_colour = '#be64ac'  # high pop / low NTL quadrant colour

for ax, (country, gdf) in zip(axes, gdfs_biv.items()):
    # highlight the energy-poor hotspot quadrant
    is_hotspot = (gdf['pop_class'] == 3) & (gdf['ntl_class'] == 1)
    colours = np.where(is_hotspot, hotspot_colour, '#dddddd')
    gdf.plot(color=colours, edgecolor='white', linewidth=0.3, ax=ax)

    n_hotspot = is_hotspot.sum()
    pct = n_hotspot / len(gdf) * 100
    ax.set_title(f'{country}\n{n_hotspot} hotspot tiles ({pct:.0f}%)',
                 fontsize=11, fontweight='bold')
    ax.set_axis_off()

fig.suptitle('Energy-Poor Hotspots: High Population Density × Low NTL\n'
             '(SDG 7 Priority Zones — Bivariate Class: High pop / Low NTL)',
             fontsize=12, fontweight='bold')
fig.tight_layout()
fig.savefig(FIGURES / 'bivariate_hotspot_sdg7.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved → figures/bivariate_hotspot_sdg7.png')